## TSNE Foote Novelty

This script uses Foote novelty as predicted topic shifts in CANDOR conversations and reduces embeddings to 2D space using TSNE.

**Author:** Helen Schmidt  
**Python version:** 3.11.13

In [10]:
# libraries
import os
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import sklearn
from sklearn import cluster
from sklearn.cluster import KMeans

In [11]:
# define data input location
input_dir = "/Users/helenschmidt/Library/CloudStorage/GoogleDrive-helenschmidt129@gmail.com/My Drive/SANLab/Experiments/Conversation-Structure/data/output/full-sample"
# define data output location
output_dir = input_dir

## Overall, split by utterances

In [12]:
# load data
df = pd.read_pickle(input_dir + '/full_sample_tile_4_turn_split_utterances.pkl')
df.head()

,transcript_id,mode,window_size,gap_size,A_start_turn,A_end_turn,B_start_turn,B_end_turn,A_raw_start,A_raw_end,B_raw_start,B_raw_end,A_utterances,B_utterances,A_embeddings,B_embeddings,cosine_similarity,euclidean_distance
0,0020a0c5-1658-4747-99c1-2839e736b481,turn,4,0,1,4,5,8,200.74,214.56,214.03,230.46,"hey I'm gone. good, how are you? Yeah. Yeah, s...",too bad. Is this your first time doing? this i...,"[[0.04145282, -0.00791257, 0.104940765, 0.1183...","[[0.03092037, -0.10058955, 0.0143238, -0.01912...",0.440620,1.057715
1,0020a0c5-1658-4747-99c1-2839e736b481,turn,4,0,5,8,9,12,214.03,230.46,231.64,244.66,too bad. Is this your first time doing? this i...,not too bad. Feel a little crappy today but hu...,"[[0.03092037, -0.10058955, 0.0143238, -0.01912...","[[0.0076957764, -0.0027738144, 0.038302507, -0...",0.276866,1.202609
2,0020a0c5-1658-4747-99c1-2839e736b481,turn,4,0,9,12,13,16,231.64,244.66,244.07,276.46,not too bad. Feel a little crappy today but hu...,the kid hasn't been feeling good and I think s...,"[[0.0076957764, -0.0027738144, 0.038302507, -0...","[[-0.042489365, 0.021653492, 0.09216961, -0.04...",0.522794,0.976940
3,0020a0c5-1658-4747-99c1-2839e736b481,turn,4,0,13,16,17,20,244.07,276.46,277.02,317.65,the kid hasn't been feeling good and I think s...,22 girls if I'm that's awesome. So I have no k...,"[[-0.042489365, 0.021653492, 0.09216961, -0.04...","[[0.037868235, -0.0063665304, 0.023072034, 0.0...",0.177692,1.282426
4,0020a0c5-1658-4747-99c1-2839e736b481,turn,4,0,17,20,21,24,277.02,317.65,317.59,333.26,22 girls if I'm that's awesome. So I have no k...,they've both been pretty good as far as stayin...,"[[0.037868235, -0.0063665304, 0.023072034, 0.0...","[[-0.019834107, -0.07536617, -0.018573657, 0.0...",0.332540,1.155388


In [13]:
# also load foote novelty data
df_foote = pd.read_pickle(input_dir + '/foote_full_sample_tile_4_turn_split_utterances.pkl')
df_foote.head()

,transcript_id,time_index,A_start_turn,foote_novelty,predicted_shift,threshold
0,0020a0c5-1658-4747-99c1-2839e736b481,0,1,0.0,0,0.639716
1,0020a0c5-1658-4747-99c1-2839e736b481,1,5,0.0,0,0.639716
2,0020a0c5-1658-4747-99c1-2839e736b481,2,9,0.0,0,0.639716
3,0020a0c5-1658-4747-99c1-2839e736b481,3,13,0.0,0,0.639716
4,0020a0c5-1658-4747-99c1-2839e736b481,4,17,0.0,0,0.639716


In [14]:
# merge on transcript_id, time_index, and A_start_turn
# only keep transcript_id, time_index, A_start_turn, A_utterances, and A_embeddings

merged_df = (
    df[['transcript_id', 'A_start_turn', 'A_utterances', 'A_embeddings']]
      .merge(
          df_foote[['transcript_id', 'time_index', 'A_start_turn', 'predicted_shift']],
          on=['transcript_id', 'A_start_turn'],
          how='left'))

merged_df.head()

,transcript_id,A_start_turn,A_utterances,A_embeddings,time_index,predicted_shift
0,0020a0c5-1658-4747-99c1-2839e736b481,1,"hey I'm gone. good, how are you? Yeah. Yeah, s...","[[0.04145282, -0.00791257, 0.104940765, 0.1183...",0.0,0.0
1,0020a0c5-1658-4747-99c1-2839e736b481,5,too bad. Is this your first time doing? this i...,"[[0.03092037, -0.10058955, 0.0143238, -0.01912...",1.0,0.0
2,0020a0c5-1658-4747-99c1-2839e736b481,9,not too bad. Feel a little crappy today but hu...,"[[0.0076957764, -0.0027738144, 0.038302507, -0...",2.0,0.0
3,0020a0c5-1658-4747-99c1-2839e736b481,13,the kid hasn't been feeling good and I think s...,"[[-0.042489365, 0.021653492, 0.09216961, -0.04...",3.0,0.0
4,0020a0c5-1658-4747-99c1-2839e736b481,17,22 girls if I'm that's awesome. So I have no k...,"[[0.037868235, -0.0063665304, 0.023072034, 0.0...",4.0,0.0


### UMAP

In [ ]:
# remove rows with missing embeddings just in case
pca_df = merged_df.dropna(subset=['A_embeddings']).copy()

# extract embeddings from pca_df 
embeddings = np.vstack(pca_df['A_embeddings'].apply(lambda x: np.array(x).squeeze()).values)

# fit PCA with 50 initialized components
pca = PCA(n_components=50, random_state=42)
reduced_embeddings = pca.fit_transform(embeddings)

# assign PCA embeddings back to rows
pca_df['pca_embeddings'] = reduced_embeddings.tolist()

# preview
pca_df.head()

# initialize UMAP 
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42)

# get UMAP on the pca_embeddings
umap_embeddings = reducer.fit_transform(
    np.stack(pca_df['pca_embeddings'].values))

# apply UMAP and store results
pca_df[['umap1', 'umap2']] = umap_embeddings

# merge with original merged_df 
df_umap = pd.merge(
    merged_df,
    pca_df[['transcript_id', 'time_index', 'A_start_turn', 'umap1', 'umap2']],
    on=['transcript_id', 'time_index', 'A_start_turn'],
    how='left'
)

# preview
df_umap.head()


/opt/anaconda3/envs/myenv-python3-11/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


,transcript_id,A_start_turn,A_utterances,A_embeddings,time_index,predicted_shift,umap1,umap2
0,0020a0c5-1658-4747-99c1-2839e736b481,1,"hey I'm gone. good, how are you? Yeah. Yeah, s...","[[0.04145282, -0.00791257, 0.104940765, 0.1183...",0.0,0.0,1.494659,3.408331
1,0020a0c5-1658-4747-99c1-2839e736b481,5,too bad. Is this your first time doing? this i...,"[[0.03092037, -0.10058955, 0.0143238, -0.01912...",1.0,0.0,0.594709,11.302336
2,0020a0c5-1658-4747-99c1-2839e736b481,9,not too bad. Feel a little crappy today but hu...,"[[0.0076957764, -0.0027738144, 0.038302507, -0...",2.0,0.0,1.689009,3.579092
3,0020a0c5-1658-4747-99c1-2839e736b481,13,the kid hasn't been feeling good and I think s...,"[[-0.042489365, 0.021653492, 0.09216961, -0.04...",3.0,0.0,3.581552,6.709514
4,0020a0c5-1658-4747-99c1-2839e736b481,17,22 girls if I'm that's awesome. So I have no k...,"[[0.037868235, -0.0063665304, 0.023072034, 0.0...",4.0,0.0,3.361819,7.119699


In [18]:
# save umap embeddings
df_umap.to_csv(output_dir + '/umap_full_sample_tile_4_turn_split_utterances.csv', index = False)

## Overall, split by sentences

In [23]:
# load data
df = pd.read_pickle(input_dir + '/full_sample_tile_4_turn_split_sentences.pkl')
df.head()

,transcript_id,speaker_id,speaker_mode,mode,window_size,gap_size,A_start_turn,A_end_turn,B_start_turn,B_end_turn,A_raw_start,A_raw_end,B_raw_start,B_raw_end,A_utterances,B_utterances,A_embeddings,B_embeddings,cosine_similarity,euclidean_distance
0,0020a0c5-1658-4747-99c1-2839e736b481,overall,overall,turn,4,0,1,4,5,8,200.74,214.56,214.03,230.46,"hey I'm gone. good, how are you? Yeah. Yeah, s...",too bad. Is this your first time doing? this i...,"[-0.036358055, -0.040273294, 0.043422665, 0.04...","[-0.028230557, -0.051923662, 0.024701986, 0.02...",0.536274,0.591127
1,0020a0c5-1658-4747-99c1-2839e736b481,overall,overall,turn,4,0,5,8,9,12,214.03,230.46,231.64,244.66,too bad. Is this your first time doing? this i...,not too bad. Feel a little crappy today but hu...,"[-0.028230557, -0.051923662, 0.024701986, 0.02...","[-0.028711766, -0.03214254, 0.025885152, 0.012...",0.517174,0.618117
2,0020a0c5-1658-4747-99c1-2839e736b481,overall,overall,turn,4,0,9,12,13,16,231.64,244.66,244.07,276.46,not too bad. Feel a little crappy today but hu...,the kid hasn't been feeling good and I think s...,"[-0.028711766, -0.03214254, 0.025885152, 0.012...","[-0.024448179, -0.008442979, 0.031297337, -0.0...",0.497826,0.628375
3,0020a0c5-1658-4747-99c1-2839e736b481,overall,overall,turn,4,0,13,16,17,20,244.07,276.46,277.02,317.65,the kid hasn't been feeling good and I think s...,22 girls if I'm that's awesome. So I have no k...,"[-0.024448179, -0.008442979, 0.031297337, -0.0...","[-0.0057626907, -0.033707336, 0.018105716, 0.0...",0.485878,0.628936
4,0020a0c5-1658-4747-99c1-2839e736b481,overall,overall,turn,4,0,17,20,21,24,277.02,317.65,317.59,333.26,22 girls if I'm that's awesome. So I have no k...,they've both been pretty good as far as stayin...,"[-0.0057626907, -0.033707336, 0.018105716, 0.0...","[-0.003213903, -0.02016028, -0.009734786, 0.03...",0.513710,0.602606


In [24]:
# also load foote novelty data
df_foote = pd.read_pickle(input_dir + '/foote_full_sample_tile_4_turn_split_sentences.pkl')
df_foote.head()

,transcript_id,time_index,A_start_turn,foote_novelty,predicted_shift,threshold
0,0020a0c5-1658-4747-99c1-2839e736b481,0,1,0.0,0,0.667243
1,0020a0c5-1658-4747-99c1-2839e736b481,1,5,0.0,0,0.667243
2,0020a0c5-1658-4747-99c1-2839e736b481,2,9,0.0,0,0.667243
3,0020a0c5-1658-4747-99c1-2839e736b481,3,13,0.0,0,0.667243
4,0020a0c5-1658-4747-99c1-2839e736b481,4,17,0.0,0,0.667243


In [25]:
merged_df = (
    df[['transcript_id', 'A_start_turn', 'A_utterances', 'A_embeddings']]
      .merge(
          df_foote[['transcript_id', 'time_index', 'A_start_turn', 'predicted_shift']],
          on=['transcript_id', 'A_start_turn'],
          how='left'))

merged_df.head()

,transcript_id,A_start_turn,A_utterances,A_embeddings,time_index,predicted_shift
0,0020a0c5-1658-4747-99c1-2839e736b481,1,"hey I'm gone. good, how are you? Yeah. Yeah, s...","[-0.036358055, -0.040273294, 0.043422665, 0.04...",0,0
1,0020a0c5-1658-4747-99c1-2839e736b481,5,too bad. Is this your first time doing? this i...,"[-0.028230557, -0.051923662, 0.024701986, 0.02...",1,0
2,0020a0c5-1658-4747-99c1-2839e736b481,9,not too bad. Feel a little crappy today but hu...,"[-0.028711766, -0.03214254, 0.025885152, 0.012...",2,0
3,0020a0c5-1658-4747-99c1-2839e736b481,13,the kid hasn't been feeling good and I think s...,"[-0.024448179, -0.008442979, 0.031297337, -0.0...",3,0
4,0020a0c5-1658-4747-99c1-2839e736b481,17,22 girls if I'm that's awesome. So I have no k...,"[-0.0057626907, -0.033707336, 0.018105716, 0.0...",4,0


### UMAP

In [26]:
# remove rows with missing embeddings just in case
pca_df = merged_df.dropna(subset=['A_embeddings']).copy()

# extract embeddings from pca_df 
embeddings = np.vstack(pca_df['A_embeddings'].apply(lambda x: np.array(x).squeeze()).values)

# fit PCA with 50 initialized components
pca = PCA(n_components=50, random_state=42)
reduced_embeddings = pca.fit_transform(embeddings)

# assign PCA embeddings back to rows
pca_df['pca_embeddings'] = reduced_embeddings.tolist()

# preview
pca_df.head()

# initialize UMAP 
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42)

# get UMAP on the pca_embeddings
umap_embeddings = reducer.fit_transform(
    np.stack(pca_df['pca_embeddings'].values))

# apply UMAP and store results
pca_df[['umap1', 'umap2']] = umap_embeddings

# merge with original merged_df 
df_umap = pd.merge(
    merged_df,
    pca_df[['transcript_id', 'time_index', 'A_start_turn', 'umap1', 'umap2']],
    on=['transcript_id', 'time_index', 'A_start_turn'],
    how='left'
)

# preview
df_umap.head()

/opt/anaconda3/envs/myenv-python3-11/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,transcript_id,A_start_turn,A_utterances,A_embeddings,time_index,predicted_shift,umap1,umap2
0,0020a0c5-1658-4747-99c1-2839e736b481,1,"hey I'm gone. good, how are you? Yeah. Yeah, s...","[-0.036358055, -0.040273294, 0.043422665, 0.04...",0,0,0.270922,3.607325
1,0020a0c5-1658-4747-99c1-2839e736b481,5,too bad. Is this your first time doing? this i...,"[-0.028230557, -0.051923662, 0.024701986, 0.02...",1,0,0.125791,6.239286
2,0020a0c5-1658-4747-99c1-2839e736b481,9,not too bad. Feel a little crappy today but hu...,"[-0.028711766, -0.03214254, 0.025885152, 0.012...",2,0,4.456112,9.689106
3,0020a0c5-1658-4747-99c1-2839e736b481,13,the kid hasn't been feeling good and I think s...,"[-0.024448179, -0.008442979, 0.031297337, -0.0...",3,0,3.554417,5.853886
4,0020a0c5-1658-4747-99c1-2839e736b481,17,22 girls if I'm that's awesome. So I have no k...,"[-0.0057626907, -0.033707336, 0.018105716, 0.0...",4,0,3.354649,5.855151


In [27]:
# save umap embeddings
df_umap.to_csv(output_dir + '/umap_full_sample_tile_4_turn_split_sentences.csv', index = False)

## Speaker, split by utterance

In [32]:
# load data
df = pd.read_pickle(input_dir + '/full_sample_tile_4_turn_split_utterances_by_speaker.pkl')
df.head()

,transcript_id,speaker_mode,mode,window_size,gap_size,A_speaker,B_speaker,A_start_turn,A_end_turn,B_start_turn,B_end_turn,A_utterances,B_utterances,A_embeddings,B_embeddings,cosine_similarity,euclidean_distance
0,0020a0c5-1658-4747-99c1-2839e736b481,by_speaker,turn,4,0,5a73899f9cdd1800017786f0,5fa072f4f4aa580b63834357,1,4,1,4,hey I'm gone. yeah I've done a few of these be...,"good, how are you? Yeah. Yeah, so this will be...","[-0.05925276, -0.038165268, 0.042457737, 0.037...","[-0.005335858, -0.05403169, 0.02566694, 0.0360...",0.472280,0.641885
1,0020a0c5-1658-4747-99c1-2839e736b481,by_speaker,turn,4,0,5fa072f4f4aa580b63834357,5a73899f9cdd1800017786f0,1,4,5,8,"good, how are you? Yeah. Yeah, so this will be...",not too bad. Feel a little crappy today but hu...,"[-0.005335858, -0.05403169, 0.02566694, 0.0360...","[-0.06539267, -0.016892936, 0.035326734, 0.000...",0.383021,0.718863
2,0020a0c5-1658-4747-99c1-2839e736b481,by_speaker,turn,4,0,5a73899f9cdd1800017786f0,5fa072f4f4aa580b63834357,5,8,5,8,not too bad. Feel a little crappy today but hu...,"I don't know. no, are you getting sick? Oh tha...","[-0.06539267, -0.016892936, 0.035326734, 0.000...","[0.012232725, -0.023692584, 0.021855772, 0.008...",0.451708,0.668791
3,0020a0c5-1658-4747-99c1-2839e736b481,by_speaker,turn,4,0,5fa072f4f4aa580b63834357,5a73899f9cdd1800017786f0,5,8,9,12,"I don't know. no, are you getting sick? Oh tha...",22 girls if I'm enjoy it while you can they've...,"[0.012232725, -0.023692584, 0.021855772, 0.008...","[0.010743233, -0.03177115, -0.0020472077, 0.03...",0.317040,0.707071
4,0020a0c5-1658-4747-99c1-2839e736b481,by_speaker,turn,4,0,5a73899f9cdd1800017786f0,5fa072f4f4aa580b63834357,9,12,9,12,22 girls if I'm enjoy it while you can they've...,that's awesome. So I have no kids myself not a...,"[0.010743233, -0.03177115, -0.0020472077, 0.03...","[-0.01709637, -0.01758504, -0.0006486526, 0.03...",0.542900,0.587993


In [33]:
# also load foote novelty data
df_foote = pd.read_pickle(input_dir + '/foote_full_sample_tile_4_turn_split_utterances_by_speaker.pkl')
df_foote.head()

,transcript_id,time_index,A_start_turn,foote_novelty,speaker,predicted_shift,threshold
0,0020a0c5-1658-4747-99c1-2839e736b481,0,1,0.0,None,0,0.513207
1,0020a0c5-1658-4747-99c1-2839e736b481,1,1,0.0,None,0,0.513207
2,0020a0c5-1658-4747-99c1-2839e736b481,2,5,0.0,None,0,0.513207
3,0020a0c5-1658-4747-99c1-2839e736b481,3,5,0.0,None,0,0.513207
4,0020a0c5-1658-4747-99c1-2839e736b481,4,9,0.0,None,0,0.513207


In [39]:
merged_df = (
    df[['transcript_id', 'A_start_turn', 'A_speaker', 'A_utterances', 'A_embeddings']]
      .merge(
          df_foote[['transcript_id', 'time_index', 'A_start_turn', 'predicted_shift']],
          on=['transcript_id', 'A_start_turn'],
          how='left'))

# remove duplicate rows
merged_df = merged_df.drop_duplicates(subset=['transcript_id', 'A_speaker', 'A_utterances', 'A_start_turn'], keep = "first")

merged_df.head()


,transcript_id,A_start_turn,A_speaker,A_utterances,A_embeddings,time_index,predicted_shift
0,0020a0c5-1658-4747-99c1-2839e736b481,1,5a73899f9cdd1800017786f0,hey I'm gone. yeah I've done a few of these be...,"[-0.05925276, -0.038165268, 0.042457737, 0.037...",0,0
2,0020a0c5-1658-4747-99c1-2839e736b481,1,5fa072f4f4aa580b63834357,"good, how are you? Yeah. Yeah, so this will be...","[-0.005335858, -0.05403169, 0.02566694, 0.0360...",0,0
4,0020a0c5-1658-4747-99c1-2839e736b481,5,5a73899f9cdd1800017786f0,not too bad. Feel a little crappy today but hu...,"[-0.06539267, -0.016892936, 0.035326734, 0.000...",2,0
6,0020a0c5-1658-4747-99c1-2839e736b481,5,5fa072f4f4aa580b63834357,"I don't know. no, are you getting sick? Oh tha...","[0.012232725, -0.023692584, 0.021855772, 0.008...",2,0
8,0020a0c5-1658-4747-99c1-2839e736b481,9,5a73899f9cdd1800017786f0,22 girls if I'm enjoy it while you can they've...,"[0.010743233, -0.03177115, -0.0020472077, 0.03...",4,0


In [41]:
# remove rows with missing embeddings just in case
pca_df = merged_df.dropna(subset=['A_embeddings']).copy()

# extract embeddings from pca_df 
embeddings = np.vstack(pca_df['A_embeddings'].apply(lambda x: np.array(x).squeeze()).values)

# fit PCA with 50 initialized components
pca = PCA(n_components=50, random_state=42)
reduced_embeddings = pca.fit_transform(embeddings)

# assign PCA embeddings back to rows
pca_df['pca_embeddings'] = reduced_embeddings.tolist()

# preview
pca_df.head()

# initialize UMAP 
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42)

# get UMAP on the pca_embeddings
umap_embeddings = reducer.fit_transform(
    np.stack(pca_df['pca_embeddings'].values))

# apply UMAP and store results
pca_df[['umap1', 'umap2']] = umap_embeddings

# merge with original merged_df 
df_umap = pd.merge(
    merged_df,
    pca_df[['transcript_id', 'time_index', 'A_speaker', 'A_start_turn', 'umap1', 'umap2']],
    on=['transcript_id', 'time_index', 'A_start_turn', 'A_speaker'],
    how='left'
)

# preview
df_umap.head()

/opt/anaconda3/envs/myenv-python3-11/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,transcript_id,A_start_turn,A_speaker,A_utterances,A_embeddings,time_index,predicted_shift,umap1,umap2
0,0020a0c5-1658-4747-99c1-2839e736b481,1,5a73899f9cdd1800017786f0,hey I'm gone. yeah I've done a few of these be...,"[-0.05925276, -0.038165268, 0.042457737, 0.037...",0,0,4.982636,3.609478
1,0020a0c5-1658-4747-99c1-2839e736b481,1,5fa072f4f4aa580b63834357,"good, how are you? Yeah. Yeah, so this will be...","[-0.005335858, -0.05403169, 0.02566694, 0.0360...",0,0,1.012898,9.158529
2,0020a0c5-1658-4747-99c1-2839e736b481,5,5a73899f9cdd1800017786f0,not too bad. Feel a little crappy today but hu...,"[-0.06539267, -0.016892936, 0.035326734, 0.000...",2,0,1.371329,7.731795
3,0020a0c5-1658-4747-99c1-2839e736b481,5,5fa072f4f4aa580b63834357,"I don't know. no, are you getting sick? Oh tha...","[0.012232725, -0.023692584, 0.021855772, 0.008...",2,0,4.439325,5.809759
4,0020a0c5-1658-4747-99c1-2839e736b481,9,5a73899f9cdd1800017786f0,22 girls if I'm enjoy it while you can they've...,"[0.010743233, -0.03177115, -0.0020472077, 0.03...",4,0,4.850161,3.976472


In [44]:
# save umap embeddings
df_umap.to_csv(output_dir + '/umap_full_sample_tile_4_turn_split_utterances_by_speaker.csv', index = False)